In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from utils import *
from interpolation import *
import pandas as pd


qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


In [7]:
files_fdt = sorted(glob.glob('/home/ulyanov/data/stereo/2025-09-24/fdt/*'))
files_fdt

['/home/ulyanov/data/stereo/2025-09-24/fdt/solo_L2_phi-fdt-bamb_20250924T061503_V202602220437_0549240502.fits.gz',
 '/home/ulyanov/data/stereo/2025-09-24/fdt/solo_L2_phi-fdt-bazi_20250924T061503_V202602220437_0549240502.fits.gz',
 '/home/ulyanov/data/stereo/2025-09-24/fdt/solo_L2_phi-fdt-binc_20250924T061503_V202602220437_0549240502.fits.gz',
 '/home/ulyanov/data/stereo/2025-09-24/fdt/solo_L2_phi-fdt-blos_20250924T061503_V202602220437_0549240502.fits.gz',
 '/home/ulyanov/data/stereo/2025-09-24/fdt/solo_L2_phi-fdt-vlos_20250924T061503_V202602220437_0549240502.fits.gz']

In [8]:
files_hmi = sorted(glob.glob('/home/ulyanov/data/stereo/2025-09-24/hmi/*'))
files_hmi

['/home/ulyanov/data/stereo/2025-09-24/hmi/hmi.M_45s.20250924_062015_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/stereo/2025-09-24/hmi/hmi.V_45s.20250924_062015_TAI.2.Dopplergram.fits']

In [3]:
df = pd.read_csv('/home/ulyanov/data/solo/phi/wcs/fdt/disk_centers.csv', skipinitialspace=True).drop(columns='date').dropna()
dids = df['did'].to_numpy()
xu_sun, yu_sun, ru_sun = df['xu_sun'].to_numpy(), df['yu_sun'].to_numpy(), df['ru_sun'].to_numpy()

s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/distortion.npz')
xd, yd = s['xd'], s['yd']

In [9]:
file_hmi = files_hmi[1]
file_fdt = files_fdt[-1]

with fits.open(file_hmi) as hdul:
    header_hmi = hdul[1].header
    data_hmi = hdul[1].data

with fits.open(file_fdt) as hdul:
    header_fdt = hdul[0].header
    data_fdt = hdul[0].data

did = int(file_fdt.split('_')[-1].split('.')[0])

data_fdt = undistort(data_fdt, header_fdt, xd, yd) #* 1.3
data_hmi = rebin(data_hmi, 8, update_header=header_hmi) / 1000

view_fdt = View.from_header(header_fdt).update(xc=xu_sun[dids == did][0], yc=yu_sun[dids == did][0], crota=header_fdt['CROTA'] + 0.26, rsun=ru_sun[dids == did][0])
view_hmi = View.from_header(header_hmi)

data_fdt -= np.nan_to_num(view_fdt.update(vr=0).velocity() / 1000)
data_hmi -= np.nan_to_num(view_hmi.velocity() / 1000)


view_new = view_fdt.update(crota=0)
transform_fdt = (view_new.to_carrington() -
                 view_fdt.to_carrington(mu_thr=0.2))


grid, alpha = transform_fdt(view_new.grid())
data_fdt = bilinear(data_fdt, *grid) * alpha

#view_new = view_fdt.update(crlt=view_hmi.crlt, crota=0)
transform_hmi = (view_new.to_carrington() -
                 view_hmi.to_carrington(mu_thr=0.2))

grid, alpha = transform_hmi(view_new.grid())
data_hmi = bilinear(data_hmi, *grid) * alpha

In [14]:
plt.figure(figsize=(10,10))
plt.imshow(data_hmi, cmap='seismic', vmin=-1, vmax=1)
plt.tight_layout()

In [13]:
plt.figure(figsize=(10,10))
plt.imshow(data_fdt, cmap='seismic', vmin=-1, vmax=1)
#plt.imshow(data_fdt - np.nan_to_num(view_new.dr_velocity() / 1000), cmap='seismic', vmin=-1, vmax=1)
plt.tight_layout()

In [16]:
transform = view_hmi.to_carrington(origin='heliographic') - view_fdt.to_carrington(origin='heliographic')

e1 = (0,0,1)
e2 = transform(e1)[0]

e1 = np.array(e1)
e2 = np.array(e2)

q = np.sum(e1 * e2)
delta = np.sqrt(1 - q ** 2)

e2_ = (e2 - e1 * q) / delta

v1 = data_fdt
v2 = data_hmi

V11 = np.sqrt((v1 ** 2 + v2 ** 2 - 2 * v1 * v2 * q)) / delta

Vq = np.sqrt(V11 ** 2 - v1 ** 2)

In [19]:
plt.figure(figsize=(10,10))
plt.imshow(V11, cmap='jet', vmin=0., vmax=5)
plt.tight_layout()

In [18]:
plt.figure(figsize=(10,10))
plt.imshow(Vq, cmap='jet', vmin=0.5, vmax=3.5)
plt.tight_layout()

In [12]:
plt.figure(figsize=(10,10))
plt.imshow(np.abs(v1), cmap='jet', vmin=0.5, vmax=3.5)
plt.tight_layout()

In [13]:
_, yi, _ = view_new.grid(origin='carrington')
y = np.arange(0,1,0.001)


r = RSUN * np.pi / 180 / 24 / 3600 / 1000
U = (A + B * y ** 2 + C * y ** 4) * r * np.sqrt(1 - y ** 2)

plt.figure(figsize=(10,10))
plt.scatter(np.abs(yi), V11, s=0.01)
plt.plot(y, U, '--', color='black')

plt.ylim(0,4)
plt.tight_layout()

In [28]:
_, yi, _ = view_new.grid(origin='carrington')


r = RSUN * np.pi / 180 / 24 / 3600 / 1000
V11_ = (A + B * yi ** 2 + C * yi ** 4) * r * np.sqrt(1 - yi ** 2)

plt.figure(figsize=(10,10))
plt.imshow(V11_, cmap='jet')
plt.tight_layout()

In [29]:
plt.figure(figsize=(10,10))
plt.imshow(V11 - V11_, cmap='bwr', vmin=-2, vmax=2)

plt.tight_layout()